# Assignment VOL

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_excel('/Users/jlaw/projects/stern/systematic-investing/data/assignment_VOL_data.xlsx')
pd.set_option('display.max_columns', None) # Show all columns when printing a dataframe
df.head()

## Part A: Setup Data and Indicators

### Step 1: Calculate Daily Returns and 20-day Volatility

In [ ]:
dfvol = df[['Date','Close']]

# Assign a new index starting from 1
dfvol.index = np.arange(1, len(dfvol) + 1)

#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)

# Compute daily returns
dfvol['dailyPct'] = dfvol['Close'].pct_change()

# Compute 20 day volatility (standard deviation of daily returns)
dfvol['vol20'] = dfvol['dailyPct'].rolling(window=20).std()

# Compute previous 20 day return (cumulative return over the past 20 days)
dfvol['ret20'] = dfvol['dailyPct'].rolling(window=20).apply(lambda x: np.prod(1 + x) - 1)

# Compute future 20 day return (cumulative return over the next 20 days)
dfvol['fret20'] = (1 + dfvol['dailyPct']).rolling(window=20).apply(np.prod, raw=True).shift(-19) - 1

### Step 2: Calculate Z-Scores (Normalized Volatility and Returns)

Using a 250-day rolling window for mean and standard deviation

In [ ]:
# Normalize vol20 to get zvol20 (backward-looking 250-day window)
dfvol['zvol20'] = (dfvol['vol20'] - dfvol['vol20'].rolling(window=250).mean()) / dfvol['vol20'].rolling(window=250).std()

# Normalize ret20 to get zret20 (backward-looking 250-day window)
dfvol['zret20'] = (dfvol['ret20'] - dfvol['ret20'].rolling(window=250).mean()) / dfvol['ret20'].rolling(window=250).std()

# Normalize fret20 to get zfret20 (backward-looking 250-day window)
dfvol['zfret20'] = (dfvol['fret20'] - dfvol['fret20'].rolling(window=250).mean()) / dfvol['fret20'].rolling(window=250).std()

### Step 3: Prepare Clean Dataset for Analysis

Remove rows with missing data and create quintile buckets

In [ ]:
# Remove rows with NaN values (due to rolling calculations)
dfvol.dropna(inplace=True)

# Sort by zvol20 and assign quintiles
dfvol = dfvol.sort_values(by='zvol20').reset_index(drop=True)
dfvol['vol_quintile'] = pd.qcut(dfvol.index, q=5, labels=False) + 1  # Quintiles labeled 1 to 5

## Part A: Analysis Questions

### Question 1: Concurrent Relationship between zvol20 and zret20

Is there a concurrent relationship between normalized volatility and normalized returns?

In [ ]:
# Group by vol_quintile and calculate average zvol20 and zret20 for each quintile
quintile_stats = dfvol.groupby('vol_quintile').agg({'zvol20': 'mean', 'zret20': 'mean'}).reset_index()
print(quintile_stats)

# Plot relationship between average zvol20 and average zret20 for each quintile
plt.plot(quintile_stats['zvol20'], quintile_stats['zret20'])
plt.xlabel('Average zvol20')
plt.ylabel('Average zret20')
plt.show()

### Question 2: Lead-Lag Relationship between zvol20 and zfret20

Is there a lead-lag relationship between normalized volatility and FUTURE normalized returns?

In [ ]:
# Group by vol_quintile and calculate average zvol20 and zfret20 for each quintile
quintile_stats = dfvol.groupby('vol_quintile').agg({'zvol20': 'mean', 'zfret20': 'mean'}).reset_index()
print(quintile_stats)

# Plot relationship between average zvol20 and average zfret20 for each quintile
plt.plot(quintile_stats['zvol20'], quintile_stats['zfret20'])
plt.xlabel('Average zvol20')
plt.ylabel('Average zfret20')
plt.show()

In [ ]:
# Correlation between zvol20 and zret20
correlation = dfvol['zvol20'].corr(dfvol['zret20'])
print(f'Correlation between zvol20 and zret20: {correlation}')

# Correlation between zvol20 and zfret20
correlation = dfvol['zvol20'].corr(dfvol['zfret20'])
print(f'Correlation between zvol20 and zfret20: {correlation}')


### There is a negative correlation between returns and volatility
#### Higher volatility drags down returns

### There is little-to-no correlation between future returns and volatility
#### Cannot use volatility to predict future returns

### Thoughts:
#### Classic -5% off 100 is 95, but +5% on 95 is only 99.75 example (volatility drag). Shows that volatility hurts returns.
#### Presumably all volatility information is priced into today's prices, so knowing volatility can't help you find an edge.

In [ ]:
#sort dfvol by Date
dfvol = dfvol.sort_values('Date')
# Take 1x daily return when zvol20 is less than -.5 stdev or greater than .5 stdev, otherwise -1x return - in a new column called 'strategy_return'
dfvol['strategy_return'] = np.where((dfvol['zvol20'] < 0.) | (dfvol['zvol20'] > 0.5), dfvol['dailyPct'], -dfvol['dailyPct'])
# Calculate cumulative return of the strategy
dfvol['cumulative_strategy_return'] = (1 + dfvol['strategy_return']).cumprod() - 1
# visualize the cumulative return of the strategy over time
plt.plot(dfvol['Date'], dfvol['cumulative_strategy_return'])
plt.xlabel('Date')
plt.ylabel('Cumulative Strategy Return')
plt.title('Cumulative Return of Volatility-Based Strategy')
plt.show()

In [ ]:
# sharpe ratio of the strategy
sharpe_ratio = ((dfvol['strategy_return'].mean() - (.037 / 252)) / dfvol['strategy_return'].std()) * np.sqrt(252)  # Annualize the Sharpe ratio
print(f'Sharpe Ratio of the strategy: {sharpe_ratio}')

In [ ]:
dfvol.head(100)